In [1]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
import requests
import time

In [2]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv('column_looks_like_paper_filtered.csv')
print(f"Loaded {len(df)} transactions, {df['From'].nunique()} unique sender addresses")

# Keep only successful transactions
df = df[df['Status'] == 'Success'].copy()
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)
df['Txn Fee'] = pd.to_numeric(df['Txn Fee'], errors='coerce').fillna(0)
df = df.sort_values('Timestamp').reset_index(drop=True)


Loaded 33385 transactions, 2993 unique sender addresses


In [3]:
# 2. FEATURE ENGINEERING 

ETHERSCAN_API_KEY = 'SKUNVYVRYG7V5IYMJJ4X9JWS6F54FZKKXH' 

def get_contract_opcodes(address):
    """
    Fetch contract bytecode from Etherscan and compute
    a simple opcode count / intensity as a numeric feature.
    """
    url = f"https://api.etherscan.io/api?module=contract&action=getsourcecode&address={address}&apikey={ETHERSCAN_API_KEY}"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        result = r.json()['result'][0]
        bytecode = result['SourceCode'] if result['SourceCode'] else result['ABI']
        # Simple proxy for opcode complexity: count length or number of instructions
        if bytecode and len(bytecode) > 0:
            opcode_count = len(bytecode)
        else:
            opcode_count = 0
        time.sleep(0.2)  # rate limit
        return opcode_count
    except:
        return 0

def extract_ttps_features(df):
    records = []

    all_addresses = set(df['From'].tolist()) | set(df['To'].tolist())

    for addr in all_addresses:
        sent = df[df['From'] == addr]
        recv = df[df['To'] == addr]

        if len(sent) == 0 and len(recv) == 0:
            continue

        # ── TTPS features ─────────────────────────
        NI = len(recv)
        NP = len(sent)

        partners = set(sent['To'].tolist()) | set(recv['From'].tolist())
        p = len(partners)
        if p >= 3:
            V = []
            for partner in partners:
                mi = len(df[(df['From'] == addr) & (df['To'] == partner)])
                ni_p = len(df[(df['From'] == partner) & (df['To'] == addr)])
                V.append(ni_p - mi)
            V = np.array(V, dtype=float)
            std = V.std()
            D_ind = float(pd.Series(V).skew()) if std > 0 else 0.0
        else:
            D_ind = 0.0

        if len(recv) > 0:
            total_recv_amount = recv['Amount'].sum()
            if total_recv_amount > 0:
                recv_sorted = recv.sort_values('Timestamp').reset_index(drop=True)
                n = len(recv_sorted)
                R_PT = float(sum(recv_sorted.loc[i, 'Amount']*((i+1)/n) for i in range(n))/total_recv_amount)
            else:
                R_PT = 0.5
        else:
            R_PT = 0.5

        T = len(sent)
        if T > 0 and len(sent['To']) > 0:
            R_mn = sent['To'].value_counts().iloc[0]
            R_MN = R_mn / T
        else:
            R_MN = 0.0

        P_total = sent['Amount'].sum()
        if P_total > 0 and len(sent) > 0:
            P_mn = sent.groupby('To')['Amount'].sum().max()
            R_MM = float(P_mn / P_total)
        else:
            R_MM = 0.0

        total_in = recv['Amount'].sum()
        total_out = sent['Amount'].sum()
        balance = total_in - total_out
        R_Bal = float(balance / total_in) if total_in > 0 else 0.0

        avg_fee = sent['Txn Fee'].mean() if len(sent) > 0 else 0.0
        tx_count = len(sent) + len(recv)

        # ── OPCODE FEATURE ───────────────────────
        opcode_feature = get_contract_opcodes(addr)

        records.append({
            'address': addr,
            'NI': NI,
            'NP': NP,
            'D_ind': D_ind,
            'R_PT': R_PT,
            'R_MN': R_MN,
            'R_MM': R_MM,
            'R_Bal': R_Bal,
            'avg_fee': avg_fee,
            'tx_count': tx_count,
            'opcode_count': opcode_feature  # NEW FEATURE
        })

    return pd.DataFrame(records)

features_df = extract_ttps_features(df)
print(f"\nExtracted features for {len(features_df)} addresses")
print(features_df.describe().round(4))



Extracted features for 5731 addresses
              NI         NP      D_ind       R_PT       R_MN       R_MM  \
count  5731.0000  5731.0000  5731.0000  5731.0000  5731.0000  5731.0000   
mean      5.7793     5.7793    -0.0006     0.5215     0.5175     0.5180   
std       9.8205     5.9261     0.0707     0.0241     0.4996     0.4996   
min       0.0000     0.0000    -4.9942     0.1416     0.0000     0.0000   
25%       0.0000     0.0000     0.0000     0.5000     0.0000     0.0000   
50%       0.0000    11.0000     0.0000     0.5000     1.0000     1.0000   
75%      11.0000    11.0000     0.0000     0.5455     1.0000     1.0000   
max     561.0000   110.0000     1.7321     0.7605     1.0000     1.0000   

             R_Bal    avg_fee   tx_count  opcode_count  
count    5731.0000  5731.0000  5731.0000        5731.0  
mean      -52.0847     0.0000    11.5585           0.0  
std      2196.0448     0.0000     8.8729           0.0  
min   -124999.0000     0.0000    11.0000           0.0  


In [4]:
# 4. NORMALISE FEATURES (add opcode_count)
# ─────────────────────────────────────────────
feature_cols = ['NI','NP','D_ind','R_PT','R_MN','R_MM','R_Bal','avg_fee','tx_count','opcode_count']
X = features_df[feature_cols].fillna(0).values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [24]:
# ─────────────────────────────────────────────
# 5. AUTOENCODER ANOMALY DETECTION
# ─────────────────────────────────────────────

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import numpy as np

input_dim = X_scaled.shape[1]

# Build Autoencoder
input_layer = Input(shape=(input_dim,))
encoder = Dense(16, activation="relu")(input_layer)
encoder = Dense(8, activation="relu")(encoder)

decoder = Dense(16, activation="relu")(encoder)
decoder = Dense(input_dim, activation="linear")(decoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

# Train
autoencoder.fit(
    X_scaled,
    X_scaled,
    epochs=30,
    batch_size=64,
    shuffle=True,
    verbose=0
)

# Reconstruction Error
reconstructions = autoencoder.predict(X_scaled)
mse = np.mean(np.power(X_scaled - reconstructions, 2), axis=1)

threshold = np.percentile(mse, 99.8)

features_df['autoencoder_score'] = mse
features_df['is_suspicious1'] = (mse > threshold).astype(int)

print(f"\nSuspicious addresses flagged: {features_df['is_suspicious1'].sum()} / {len(features_df)}")

# ── Summary table ─────────────────────
print("\n── Predicted Suspicious Addresses ─────────────────────────────")

summary_autoencoder = features_df[features_df['is_suspicious1'] == 1][
    ['address','autoencoder_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='autoencoder_score', ascending=False).reset_index(drop=True)
summary_autoencoder.index += 1

print(summary_autoencoder.to_string())

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 559us/step

Suspicious addresses flagged: 12 / 5731

── Predicted Suspicious Addresses ─────────────────────────────
                                       address  autoencoder_score   NI   NP      R_PT      R_MN      R_MM          R_Bal
1   0x45300136662dd4e58fc0df61e6290dffd992b785           0.216867  561   22  0.500891  0.500000  0.930233     -41.156863
2   0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f           0.124658  132  110  0.692649  0.100000  0.670196       0.897104
3   0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23           0.072818   44   77  0.514392  0.142857  0.909091    -188.655237
4   0x28c6c06298d514db089934071355e5743bf21d60           0.061414   99   66  0.631153  0.166667  0.959269       0.900548
5   0x1111111254eeb25477b68fb85ed929f73a960582           0.051875   44    0  0.141633  0.000000  0.000000       1.000000
6   0xd9df09b252db30f7881ff5cad96b7efabf6fe745           0.049764   11   11  0.545455  1.000000  1.000000 -124999.000000
7   

In [45]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────
# 1. ANOMALY SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['autoencoder_score'], bins=50)
plt.title("Anomaly Score Distribution For Autoencoder",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.5,0.2)
plt.ylim(0,6000)
plt.savefig("anomaly_score_AE.png",dpi=600)
plt.close()

print("\nSaved: anomaly_score_distribution.png")

# ─────────────────────────────────────────────
# 2. TOP-K SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
top_k = features_df.sort_values(by='autoencoder_score', ascending=False).head(7)

print(f"\n── Top {7} Most Suspicious Addresses ─────────────────────────")
print(top_k[['address','autoencoder_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))

# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (2D)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1'] = X_pca[:,0]
features_df['pca2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1'],
    features_df['pca2'],
    c=features_df['is_suspicious1'],
)
plt.title("PCA Projection Of Autoencoder Detected Samples",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_AE.png",dpi=600)
plt.close()

print("Saved: pca_anomaly.png")

# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (QUALITY CHECK)
# ─────────────────────────────────────────────
# Convert anomaly labels to 0/1 clusters
labels = features_df['is_suspicious1'].values

# Only compute if both classes exist
if len(np.unique(labels)) > 1:
    sil_score = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette Score: {sil_score:.4f}")
else:
    print("\nSilhouette Score: Not applicable (only one class found)")


Saved: anomaly_score_distribution.png

── Top 7 Most Suspicious Addresses ─────────────────────────
                                   address  autoencoder_score  NI  NP     R_PT     R_MN     R_MM          R_Bal
0x45300136662dd4e58fc0df61e6290dffd992b785           0.216867 561  22 0.500891 0.500000 0.930233     -41.156863
0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f           0.124658 132 110 0.692649 0.100000 0.670196       0.897104
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23           0.072818  44  77 0.514392 0.142857 0.909091    -188.655237
0x28c6c06298d514db089934071355e5743bf21d60           0.061414  99  66 0.631153 0.166667 0.959269       0.900548
0x1111111254eeb25477b68fb85ed929f73a960582           0.051875  44   0 0.141633 0.000000 0.000000       1.000000
0xd9df09b252db30f7881ff5cad96b7efabf6fe745           0.049764  11  11 0.545455 1.000000 1.000000 -124999.000000
0x4976a4a02f38326660d17bf34b431dc6e2eb2327           0.049429   0  77 0.500000 0.142857 0.549882       0.000000
Sav

In [26]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['is_suspicious1'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")



─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 12
Total: 5731
Ratio: 0.00209


In [9]:
df3 = pd.read_csv('column_looks_like_paper_filtered.csv')
df4=pd.read_csv("all_labeled_data_final.csv")

In [10]:
same_values = set(summary_autoencoder['address']).intersection(set(df4['data']))
print(same_values)

{'0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x03f34be1bf910116595db1b11e9d1b2ca5d59659', '0x1111111254eeb25477b68fb85ed929f73a960582', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', '0x45300136662dd4e58fc0df61e6290dffd992b785', '0x3a23f943181408eac424116af7b7790c94cb97a5', '0x28c6c06298d514db089934071355e5743bf21d60', '0x111111125421ca6dc452d289314280a0f8842a65', '0x4976a4a02f38326660d17bf34b431dc6e2eb2327', '0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23'}


In [29]:
# ─────────────────────────────────────────────
# 6. AUTOENCODER + SMOTE
# ─────────────────────────────────────────────

from imblearn.over_sampling import SMOTE

# Use autoencoder labels
y_smote = features_df['is_suspicious1'].values

# Apply SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote_resampled = smote.fit_resample(X_scaled, y_smote)

# Train Autoencoder again
input_layer = Input(shape=(input_dim,))
encoder = Dense(16, activation="relu")(input_layer)
encoder = Dense(8, activation="relu")(encoder)

decoder = Dense(16, activation="relu")(encoder)
decoder = Dense(input_dim, activation="linear")(decoder)

autoencoder_smote = Model(inputs=input_layer, outputs=decoder)
autoencoder_smote.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

autoencoder_smote.fit(
    X_smote,
    X_smote,
    epochs=30,
    batch_size=64,
    shuffle=True,
    verbose=0
)

# Score original data
reconstruction_smote = autoencoder_smote.predict(X_scaled)
mse_smote = np.mean(np.power(X_scaled - reconstruction_smote, 2), axis=1)

threshold_smote = np.percentile(mse_smote, 90)

features_df['autosmote_score'] = mse_smote
features_df['autosmote_flag'] = (mse_smote > threshold_smote).astype(int)

print(f"\nSMOTE Suspicious: {features_df['autosmote_flag'].sum()} / {len(features_df)}")

# Summary
autoencoder_smote_summary = features_df[features_df['autosmote_flag'] == 1][
    ['address','autosmote_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='autosmote_score', ascending=False).reset_index(drop=True)

autoencoder_smote_summary.index += 1

print("\n── Autoencoder + SMOTE Suspicious ─────────────────────────")
print(autoencoder_smote_summary.to_string())

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step

SMOTE Suspicious: 280 / 5731

── Autoencoder + SMOTE Suspicious ─────────────────────────
                                        address  autosmote_score   NI   NP      R_PT      R_MN      R_MM          R_Bal
1    0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2         0.002216  110    0  0.410125  0.000000  0.000000       1.000000
2    0xc16157e00b1bff1522c6f01246b4fb621da048d0         0.000964    0   22  0.500000  0.500000  0.666667       0.000000
3    0xf179522c2f4603ae729833958546b7269f4238f9         0.000752    0   22  0.500000  1.000000  1.000000       0.000000
4    0xf8b5caa0bb0288463afd2f4f22df5d51123c4c25         0.000752    0   22  0.500000  1.000000  1.000000       0.000000
5    0xe2c08e3891608045f7274df8c616d99e5bacba34         0.000752    0   22  0.500000  1.000000  1.000000       0.000000
6    0xbc04655fdb6db273ce7faf7e62a6ec8676f72cf5         0.000752    0   22  0.500000  1.000000  1.000000       0.000000
7    0x9b23481f79574007e180

In [50]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────
# 1. ANOMALY SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['autosmote_score'], bins=50)
plt.title("Anomaly Score Distribution For AE+SMOTE",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.0010,0.0010)
plt.ylim(0,6000)
plt.savefig("anomaly_score_AE+SMOTE.png",dpi=600)
plt.close()

print("\nSaved: anomaly_score_distribution.png")

# ─────────────────────────────────────────────
# 2. TOP-K SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
top_k = features_df.sort_values(by='autosmote_score', ascending=False).head(7)

print(f"\n── Top {7} Most Suspicious Addresses ─────────────────────────")
print(top_k[['address','autosmote_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))

# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (2D)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1'] = X_pca[:,0]
features_df['pca2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1'],
    features_df['pca2'],
    c=features_df['autosmote_flag'],
)
plt.title("PCA Projection Of AE+SMOTE Detected Samples",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_AE+SMOTE.png",dpi=600)
plt.close()

print("Saved: pca_anomaly.png")

# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (QUALITY CHECK)
# ─────────────────────────────────────────────
# Convert anomaly labels to 0/1 clusters
labels = features_df['autosmote_flag'].values

# Only compute if both classes exist
if len(np.unique(labels)) > 1:
    sil_score = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette Score: {sil_score:.4f}")
else:
    print("\nSilhouette Score: Not applicable (only one class found)")


Saved: anomaly_score_distribution.png

── Top 7 Most Suspicious Addresses ─────────────────────────
                                   address  autosmote_score  NI  NP     R_PT  R_MN     R_MM  R_Bal
0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2         0.002216 110   0 0.410125   0.0 0.000000    1.0
0xc16157e00b1bff1522c6f01246b4fb621da048d0         0.000964   0  22 0.500000   0.5 0.666667    0.0
0xf179522c2f4603ae729833958546b7269f4238f9         0.000752   0  22 0.500000   1.0 1.000000    0.0
0xe2c08e3891608045f7274df8c616d99e5bacba34         0.000752   0  22 0.500000   1.0 1.000000    0.0
0xf8b5caa0bb0288463afd2f4f22df5d51123c4c25         0.000752   0  22 0.500000   1.0 1.000000    0.0
0xbc04655fdb6db273ce7faf7e62a6ec8676f72cf5         0.000752   0  22 0.500000   1.0 1.000000    0.0
0x9b23481f79574007e180c6f7435401f690787e51         0.000752   0  22 0.500000   1.0 1.000000    0.0
Saved: pca_anomaly.png

Silhouette Score: 0.1333


In [34]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['autosmote_flag'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")



─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 280
Total: 5731
Ratio: 0.04886


In [13]:
same_values = set(autoencoder_smote_summary['address']).intersection(set(df4['data']))
print(same_values)

{'0xc16157e00b1bff1522c6f01246b4fb621da048d0', '0x03f34be1bf910116595db1b11e9d1b2ca5d59659', '0x1111111254eeb25477b68fb85ed929f73a960582', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', '0x4976a4a02f38326660d17bf34b431dc6e2eb2327', '0x3a23f943181408eac424116af7b7790c94cb97a5', '0x28c6c06298d514db089934071355e5743bf21d60', '0x111111125421ca6dc452d289314280a0f8842a65', '0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x45300136662dd4e58fc0df61e6290dffd992b785'}


In [14]:
# 7. AUTOENCODER + XGBOOST
# ─────────────────────────────────────────────

from xgboost import XGBClassifier

# Use Autoencoder labels
y_xgb = features_df['is_suspicious'].values

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_scaled, y_xgb)

xgb_prob = xgb.predict_proba(X_scaled)[:,1]
xgb_pred = (xgb_prob > 0.5).astype(int)

features_df['xgb_score'] = xgb_prob
features_df['xgb_flag'] = xgb_pred

print(f"\nXGBoost Suspicious: {features_df['xgb_flag'].sum()} / {len(features_df)}")

# Summary
xgb_summary = features_df[features_df['xgb_flag'] == 1][
    ['address','xgb_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='xgb_score', ascending=False).reset_index(drop=True)

xgb_summary.index += 1

print("\n── Autoencoder + XGBoost Suspicious ──────────────────────")
print(xgb_summary.to_string())


XGBoost Suspicious: 7 / 5731

── Autoencoder + XGBoost Suspicious ──────────────────────
                                      address  xgb_score   NI   NP      R_PT      R_MN      R_MM       R_Bal
1  0x45300136662dd4e58fc0df61e6290dffd992b785   0.963439  561   22  0.500891  0.500000  0.930233  -41.156863
2  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f   0.943947  132  110  0.692649  0.100000  0.670196    0.897104
3  0x28c6c06298d514db089934071355e5743bf21d60   0.941860   99   66  0.631153  0.166667  0.959269    0.900548
4  0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23   0.918084   44   77  0.514392  0.142857  0.909091 -188.655237
5  0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2   0.913884  110    0  0.410125  0.000000  0.000000    1.000000
6  0x1111111254eeb25477b68fb85ed929f73a960582   0.913884   44    0  0.141633  0.000000  0.000000    1.000000
7  0x4976a4a02f38326660d17bf34b431dc6e2eb2327   0.558505    0   77  0.500000  0.142857  0.549882    0.000000


In [53]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────
# 1. ANOMALY SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['xgb_score'], bins=50)
plt.title("Anomaly Score Distribution For AE+XGBoost",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.5,0.2)
plt.ylim(0,6000)
plt.savefig("anomaly_score_AE+XGBoost.png",dpi=600)
plt.close()

print("\nSaved: anomaly_score_distribution.png")

# ─────────────────────────────────────────────
# 2. TOP-K SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
top_k = features_df.sort_values(by='xgb_score', ascending=False).head(7)

print(f"\n── Top {7} Most Suspicious Addresses ─────────────────────────")
print(top_k[['address','xgb_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))

# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (2D)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1'] = X_pca[:,0]
features_df['pca2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1'],
    features_df['pca2'],
    c=features_df['xgb_flag'],
)
plt.title("PCA Projection Of AE+XGBoost Detected Samples",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_AE+XGBoost.png",dpi=600)
plt.close()

print("Saved: pca_anomaly.png")

# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (QUALITY CHECK)
# ─────────────────────────────────────────────
# Convert anomaly labels to 0/1 clusters
labels = features_df['xgb_flag'].values

# Only compute if both classes exist
if len(np.unique(labels)) > 1:
    sil_score = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette Score: {sil_score:.4f}")
else:
    print("\nSilhouette Score: Not applicable (only one class found)")


Saved: anomaly_score_distribution.png

── Top 7 Most Suspicious Addresses ─────────────────────────
                                   address  xgb_score  NI  NP     R_PT     R_MN     R_MM       R_Bal
0x45300136662dd4e58fc0df61e6290dffd992b785   0.963439 561  22 0.500891 0.500000 0.930233  -41.156863
0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f   0.943947 132 110 0.692649 0.100000 0.670196    0.897104
0x28c6c06298d514db089934071355e5743bf21d60   0.941860  99  66 0.631153 0.166667 0.959269    0.900548
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23   0.918084  44  77 0.514392 0.142857 0.909091 -188.655237
0x1111111254eeb25477b68fb85ed929f73a960582   0.913884  44   0 0.141633 0.000000 0.000000    1.000000
0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2   0.913884 110   0 0.410125 0.000000 0.000000    1.000000
0x4976a4a02f38326660d17bf34b431dc6e2eb2327   0.558505   0  77 0.500000 0.142857 0.549882    0.000000
Saved: pca_anomaly.png

Silhouette Score: 0.3991


In [38]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['xgb_flag'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")



─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 7
Total: 5731
Ratio: 0.00122


In [39]:
same_values = set(xgb_summary['address']).intersection(set(df4['data']))
print(same_values)

{'0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x1111111254eeb25477b68fb85ed929f73a960582', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', '0x45300136662dd4e58fc0df61e6290dffd992b785', '0x28c6c06298d514db089934071355e5743bf21d60', '0x4976a4a02f38326660d17bf34b431dc6e2eb2327', '0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23'}


In [41]:
results = []

# ─────────────────────────────────────────────
# 1. AUTOENCODER
# ─────────────────────────────────────────────
labels_auto = features_df['is_suspicious'].values

if len(np.unique(labels_auto)) > 1:
    sil_auto = silhouette_score(X_scaled, labels_auto)
else:
    sil_auto = np.nan

ratio_auto = features_df['is_suspicious'].sum() / len(features_df)

results.append([
    "Autoencoder",
    sil_auto,
    ratio_auto,
    features_df['is_suspicious'].sum()
])


# ─────────────────────────────────────────────
# 2. AUTOENCODER + SMOTE
# ─────────────────────────────────────────────
labels_smote = features_df['autosmote_flag'].values

if len(np.unique(labels_smote)) > 1:
    sil_smote = silhouette_score(X_scaled, labels_smote)
else:
    sil_smote = np.nan

ratio_smote = features_df['autosmote_flag'].sum() / len(features_df)

results.append([
    "Autoencoder + SMOTE",
    sil_smote,
    ratio_smote,
    features_df['autosmote_flag'].sum()
])


# ─────────────────────────────────────────────
# 3. AUTOENCODER + XGBOOST
# ─────────────────────────────────────────────
labels_xgb = features_df['xgb_flag'].values

if len(np.unique(labels_xgb)) > 1:
    sil_xgb = silhouette_score(X_scaled, labels_xgb)
else:
    sil_xgb = np.nan

ratio_xgb = features_df['xgb_flag'].sum() / len(features_df)

results.append([
    "Autoencoder + XGBoost",
    sil_xgb,
    ratio_xgb,
    features_df['xgb_flag'].sum()
])


# ─────────────────────────────────────────────
# Create Comparison Table
# ─────────────────────────────────────────────
comparison_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "Silhouette Score",
        "Contamination Ratio",
        "Suspicious Count"
    ]
)

# Format
comparison_df["Silhouette Score"] = comparison_df["Silhouette Score"].round(4)
comparison_df["Contamination Ratio"] = comparison_df["Contamination Ratio"].round(5)

print("\n──────────── Model Comparison Table ────────────")
print(comparison_df.to_string(index=False))


──────────── Model Comparison Table ────────────
               Method  Silhouette Score  Contamination Ratio  Suspicious Count
          Autoencoder            0.3640              0.00209                12
  Autoencoder + SMOTE            0.1333              0.04886               280
Autoencoder + XGBoost            0.3991              0.00122                 7
